# silver_curated_hr_full

Build the silver `employee` table from the mirrored `dbo.employees` + `dbo.job_titles` + `dbo.stores` shortcuts in `contoso_retail_silver_raw`.

Pattern matches `silver_curated_retail_full`:
- read shortcuts via abfss + delta
- denormalize (resolve job_title -> title/department/level, store -> store_name/city/state, manager_id -> manager_name + manager_title)
- derive a few useful attrs (age, tenure_days, tenure_years, full_name, is_terminated, salary_band)
- write to `Tables/dbo/employee` in `contoso_retail_silver_curated`

In [ ]:
# Parameters (overridden by the pipeline at runtime via deploy.ps1)
silver_raw_workspace_id     = ""
silver_raw_lakehouse_id     = ""
silver_curated_workspace_id = ""
silver_curated_lakehouse_id = ""

AS_OF_DATE = None  # None -> current_date(); override with 'YYYY-MM-DD' for deterministic testing

In [ ]:
from pyspark.sql import functions as F

raw_base     = f"abfss://{silver_raw_workspace_id}@onelake.dfs.fabric.microsoft.com/{silver_raw_lakehouse_id}/Tables/dbo"
curated_base = f"abfss://{silver_curated_workspace_id}@onelake.dfs.fabric.microsoft.com/{silver_curated_lakehouse_id}/Tables/dbo"

def read_raw(name):
    return spark.read.format('delta').load(f'{raw_base}/{name}')

def write_curated(df, name):
    (df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(f'{curated_base}/{name}'))
    print(f'  wrote dbo.{name}  ({df.count():,} rows, {len(df.columns)} cols)')

as_of = F.current_date() if AS_OF_DATE is None else F.to_date(F.lit(AS_OF_DATE))

## Load source tables

In [ ]:
employees_src  = read_raw('employees')
job_titles_src = read_raw('job_titles')
stores_src     = read_raw('stores')

for n, df in [('employees', employees_src), ('job_titles', job_titles_src), ('stores', stores_src)]:
    print(f'{n:12s}  {df.count():>6,} rows')

## Build `employee`

Two-pass build (manager_name is a self-join), so we materialize a small `employee_name_lookup` first and join it twice -- once for the employee row, once aliased as the manager.

In [ ]:
# Minimal lookup for the self-join (employee_id -> name + title)
name_lookup = (employees_src
    .join(job_titles_src.select(F.col('job_title_id'), F.col('title').alias('_jt_title')), 'job_title_id', 'left')
    .select(
        F.col('employee_id').alias('_m_employee_id'),
        F.concat_ws(' ', 'first_name', 'last_name').alias('_m_name'),
        F.col('_jt_title').alias('_m_title'),
    ))

# Store attrs we want to carry on the employee row
stores_lite = stores_src.select(
    F.col('store_id'),
    F.col('store_name'),
    F.col('city').alias('store_city'),
    F.col('state').alias('store_state'),
)

# Job title attrs. Source schema: (job_title_id, title, department, job_level, min_salary, max_salary, is_store_role)
job_titles_lite = job_titles_src.select(
    F.col('job_title_id'),
    F.col('title'),
    F.col('department'),
    F.col('job_level').alias('level'),
)

employee = (employees_src
    .join(job_titles_lite, 'job_title_id', 'left')
    .join(stores_lite, 'store_id', 'left')
    .join(name_lookup, employees_src['manager_id'] == name_lookup['_m_employee_id'], 'left')
    .withColumn('full_name',     F.concat_ws(' ', 'first_name', 'last_name'))
    .withColumn('age',           (F.datediff(as_of, F.col('date_of_birth')) / F.lit(365.25)).cast('int'))
    .withColumn('tenure_days',   F.datediff(F.coalesce(F.col('termination_date'), as_of), F.col('hire_date')))
    .withColumn('tenure_years',  (F.col('tenure_days') / F.lit(365.25)).cast('decimal(5,2)'))
    .withColumn('is_terminated', F.col('termination_date').isNotNull())
    .withColumn('is_hq',         F.col('store_id').isNull())
    .withColumn('salary_band',
        F.when(F.col('annual_salary') <  50000, F.lit('<50k'))
         .when(F.col('annual_salary') < 100000, F.lit('50-100k'))
         .when(F.col('annual_salary') < 150000, F.lit('100-150k'))
         .when(F.col('annual_salary') < 250000, F.lit('150-250k'))
         .otherwise(F.lit('250k+')))
    .withColumn('age_band',
        F.when(F.col('age') < 25, F.lit('<25'))
         .when(F.col('age') < 35, F.lit('25-34'))
         .when(F.col('age') < 45, F.lit('35-44'))
         .when(F.col('age') < 55, F.lit('45-54'))
         .when(F.col('age') < 65, F.lit('55-64'))
         .otherwise(F.lit('65+')))
    .withColumnRenamed('_m_name',  'manager_name')
    .withColumnRenamed('_m_title', 'manager_title')
    .select(
        # identity
        'employee_id', 'first_name', 'last_name', 'full_name', 'email', 'phone',
        # demographics
        'date_of_birth', 'age', 'age_band', 'gender',
        # role
        'job_title_id', 'title', 'department', 'level',
        # location
        'store_id', 'store_name', 'store_city', 'store_state', 'is_hq',
        # hierarchy
        'manager_id', 'manager_name', 'manager_title',
        # employment lifecycle
        'hire_date', 'termination_date', 'is_active', 'is_terminated',
        'tenure_days', 'tenure_years',
        # comp
        F.col('annual_salary').cast('decimal(12,2)').alias('annual_salary'),
        'salary_band',
        # audit
        'created_at',
    ))

write_curated(employee, 'employee')

In [ ]:
print('done -- silver HR written to contoso_retail_silver_curated.dbo.employee')